# CUDA-graph batched SQP versus SciPy SLSQP

This benchmark compares SciPy SLSQP with a Torch implementation of dense, bound-constrained SQP. Both use the same five-day one-room model, canonical model-default start, parameter bounds, measurements, float64 objective, and exact first-order derivatives. The custom arm also runs seven deterministic starts sampled across the complete normalized bounds; start slot 0 is always the canonical SLSQP start and is reported separately in the multistart audit.

The Torch solver maintains a positive-definite Powell-damped BFGS Hessian, solves each box-constrained quadratic subproblem with a batched active set, and uses projected-gradient KKT convergence. All geometric line-search candidates are evaluated together in one fixed-shape batch. Direct CUDA Graph replay captures the expensive objective and first-order-gradient rollouts; only the small active-set and acceptance logic remains eager.

This supersedes the exploratory projected-BFGS/LM/Newton comparison. Those methods were rejected: projected BFGS mishandled active bounds, while eager residual Jacobians made LM roughly 29x slower than SLSQP. CUDA setup/capture time remains included in end-to-end wall time.

In [ ]:
# Colab setup. Install the exact branch through pip, then keep a checkout for the runner.
import os, subprocess, sys

REPO = "https://github.com/JBjoernskov/Twin4Build.git"
BRANCH = "feature/issue-128/collocation-initialization"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    f"git+{REPO}@{BRANCH}",
], check=True)
if os.path.exists("/content/Twin4Build"):
    subprocess.run(["git", "fetch", "--quiet", "origin", BRANCH], cwd="/content/Twin4Build", check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd="/content/Twin4Build", check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/Twin4Build"], check=True)
os.chdir("/content/Twin4Build")

import torch
print("torch", torch.__version__)
print("CUDA", torch.cuda.is_available(), torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")

In [ ]:
# CPU: use HOURS=6, MAXITER=5, N_STARTS=1.
# A100 full run: five days, the canonical start plus seven full-bound starts.
import json, pathlib, subprocess, sys

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HOURS = 120 if DEVICE == "cuda" else 6
MAXITER = 300 if DEVICE == "cuda" else 5
N_STARTS = 8 if DEVICE == "cuda" else 1
BATCH_SIZE = 4 if DEVICE == "cuda" else 1
ARMS = ["slsqp", "batched-sqp"]

rows = []
for arm in ARMS:
    output = pathlib.Path(f"/content/{arm}_{DEVICE}.json")
    if output.exists():
        print("Reusing completed", arm, "from", output)
        rows.append(json.loads(output.read_text()))
        continue
    command = [
        sys.executable, "-m", "twin4build.examples.batched_shooting_solver_benchmark",
        "--arm", arm, "--hours", str(HOURS), "--maxiter", str(MAXITER),
        "--device", DEVICE, "--output", str(output),
    ]
    if arm != "slsqp":
        command += ["--n-starts", str(N_STARTS), "--batch-size", str(BATCH_SIZE)]
        if DEVICE == "cuda":
            command += ["--capture"]
    print("Running", arm)
    completed = subprocess.run(command, capture_output=True, text=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        print(f"{arm} FAILED with exit code {completed.returncode}")
        print(completed.stderr[-12000:])
        rows.append({
            "arm": arm, "device": DEVICE, "hardware": torch.cuda.get_device_name() if DEVICE == "cuda" else "CPU",
            "hours": HOURS, "maxiter": MAXITER, "n_starts": 1 if arm == "slsqp" else N_STARTS,
            "seconds": float("nan"), "score_seconds": float("nan"), "success": False,
            "iterations": None, "objective": float("nan"), "rollout_weighted_mse": float("nan"),
            "sensor_rmse": {}, "peak_cuda_memory_gb": float("nan"),
            "error": completed.stderr[-12000:],
        })
        continue
    rows.append(json.loads(output.read_text()))
print("Finished", len(rows), "arms")

In [ ]:
import pandas as pd

summary_columns = [
    "arm", "device", "hardware", "hours", "maxiter", "n_starts",
    "seconds", "score_seconds", "success", "iterations", "objective",
    "rollout_weighted_mse", "sensor_rmse", "peak_cuda_memory_gb",
]
display(pd.DataFrame(rows)[summary_columns])

slsqp = next(row for row in rows if row["arm"] == "slsqp")
for row in rows:
    if row["arm"] == "slsqp":
        continue
    canonical = (row.get("multistart_audit") or [{}])[0]
    print(
        row["arm"],
        "speedup_vs_slsqp=", slsqp["seconds"] / row["seconds"],
        "best_objective_delta=", row["objective"] - slsqp["objective"],
        "canonical_start=", canonical,
    )

print("\nInterpretation rules:")
print("1. Compare SLSQP against SQP start slot 0 for algorithmic parity.")
print("2. Reject a faster result if it fails KKT convergence or has worse rollout quality.")
print("3. Compare cold/end-to-end and warmed derivative costs separately.")
print("4. Treat best-of-eight as a separate multistart throughput result.")

In [ ]:
derivative_rows = []
for row in rows:
    for bundle, stat in (row.get("derivative_stats") or {}).items():
        derivative_rows.append({
            "arm": row["arm"],
            "bundle": bundle,
            **stat,
            "warmed_mean_seconds": (
                (stat["seconds"] - stat["first_seconds"]) / (stat["calls"] - 1)
                if stat["calls"] > 1 else float("nan")
            ),
        })
if derivative_rows:
    display(pd.DataFrame(derivative_rows))